# Stage 4: Data Understanding & Profiling
## Student Performance Analyzer (HCL Career Shaper Program)

### Objectives
1. Ingest the raw UCI Student Performance dataset (`student-mat.csv`).
2. Verify shape, column schema, and data types.
3. Conduct statistical profiling across numeric and categorical features.
4. Audit missing values, duplicate records, and impossible values.
5. Analyze the ground-truth target variable (`G3` and derived `pass` status).
6. **Perform rigorous Target Leakage Analysis** examining period grades ($G1, G2$).
7. Audit sensitive demographic attributes for fairness implications.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure reproducible figures
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style="whitegrid")


### 1. Ingest Raw Dataset
The dataset uses semicolon (`;`) as field delimiter and contains student performance data for Mathematics.


In [ ]:
data_path = Path("../data/raw/student-mat.csv")
assert data_path.is_file(), f"Dataset not found at {data_path.resolve()}"

df = pd.read_csv(data_path, sep=";")
print(f"Raw Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


### 2. Schema and Data Types Audit

In [ ]:
df.info()


### 3. Missing Value and Duplicate Verification
The UCI documentation states that the curated dataset has no missing values. Let us verify empirically.


In [ ]:
null_counts = df.isnull().sum()
print("Total missing values across all columns:", null_counts.sum())
duplicate_rows = df.duplicated().sum()
print("Total duplicate rows detected:", duplicate_rows)


### 4. Statistical Summary of Numerical Attributes

In [ ]:
df.describe().T


### 5. Categorical Attributes and Cardinality Audit

In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
print(f"Total Categorical Columns: {len(cat_cols)}\n")
for col in cat_cols:
    uniques = df[col].unique()
    print(f"Column '{col}' ({len(uniques)} unique values): {uniques}")


### 6. Target Variable Analysis: Final Grade ($G3$) & Binary Pass/Fail
In the Portuguese secondary grading system (0 to 20):
- Passing Grade: $G3 \ge 10$
- Failing / At-Risk: $G3 < 10$


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of G3 continuous grade
sns.histplot(df["G3"], bins=20, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].axvline(10, color="red", linestyle="--", linewidth=2, label="Pass Threshold (10)")
axes[0].set_title("Distribution of Final Grade (G3)")
axes[0].set_xlabel("Final Grade (0-20)")
axes[0].set_ylabel("Student Count")
axes[0].legend()

# Pass vs Fail Distribution
pass_series = (df["G3"] >= 10).map({True: "Pass (>=10)", False: "Fail / At-Risk (<10)"})
counts = pass_series.value_counts()
colors = ["#2ecc71", "#e74c3c"]
axes[1].pie(counts, labels=counts.index, autopct="%1.1f%%", colors=colors, startangle=140, explode=(0.05, 0))
axes[1].set_title("Target Class Proportions (Pass vs. Fail)")

plt.tight_layout()
plt.show()

print("Class Distribution Breakdown:")
print(counts)
print(f"Pass Rate: {(df['G3'] >= 10).mean():.2%}")


### 7. CRITICAL REQUIREMENT: Target Leakage Analysis
The dataset contains three grade periods:
- `G1`: First period grade
- `G2`: Second period grade
- `G3`: Final grade (target)

Let us analyze the Pearson correlation between $G1, G2,$ and $G3$.


In [ ]:
grade_corr = df[["G1", "G2", "G3"]].corr()
print("Correlation Matrix between Period Grades:")
print(grade_corr)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(grade_corr, annot=True, cmap="Blues", fmt=".3f", ax=ax)
ax.set_title("Period Grade Correlation Matrix (G1, G2, G3)")
plt.show()


#### Target Leakage Evaluation & Decision
- **Observation:** $G2$ has an extremely high correlation of $r = 0.905$ with $G3$, and $G1$ has $r = 0.801$.
- **Early-Warning Dilemma:** In real-world educational deployment, teachers need risk predictions at the beginning of the term (or very early in the semester) before period examinations occur. If a model relies on $G1$ or $G2$, it merely acts as a proxy grade calculator rather than an early intervention warning system.
- **Decision:** In accordance with the Project Charter and Early-Warning requirement, **$G1$ and $G2$ are strictly EXCLUDED from all feature sets** in our predictive model. Only pre-outcome demographic, behavioral, and academic background features will be used.


### 8. Sensitive Attributes & Algorithmic Fairness Audit
The dataset includes demographic factors such as student sex (`sex`), age (`age`), residential address type (`address`), and school (`school`).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Pass rate by Sex
df["pass_binary"] = (df["G3"] >= 10).astype(int)
df.groupby("sex")["pass_binary"].mean().plot(kind="bar", ax=axes[0], color=["#9b59b6", "#3498db"], rot=0)
axes[0].set_title("Pass Rate by Sex")
axes[0].set_ylabel("Pass Rate")
axes[0].set_ylim(0, 1.0)

# Pass rate by School
df.groupby("school")["pass_binary"].mean().plot(kind="bar", ax=axes[1], color=["#1abc9c", "#f39c12"], rot=0)
axes[1].set_title("Pass Rate by School")
axes[1].set_ylabel("Pass Rate")
axes[1].set_ylim(0, 1.0)

# Pass rate by Address
df.groupby("address")["pass_binary"].mean().plot(kind="bar", ax=axes[2], color=["#34495e", "#e67e22"], rot=0)
axes[2].set_title("Pass Rate by Address (Urban vs Rural)")
axes[2].set_ylabel("Pass Rate")
axes[2].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()


### Key Findings (Stage 4)
1. **Dimensions & Completeness:** The dataset comprises exactly 395 students across 33 columns with 0 missing values and 0 duplicate rows.
2. **Target Balance:** 265 students (67.1%) passed ($G3 \ge 10$) while 130 students (32.9%) failed or are at-risk ($G3 < 10$). This class imbalance requires recall-focused evaluation.
3. **Leakage Isolation:** $G1$ and $G2$ show collinearity ($r > 0.80$) with $G3$. To guarantee true early-warning capability, both are quarantined and excluded from model inputs.
4. **Demographic Disparities:** Mild differences in pass rates appear across address types and schools, underscoring the need for fairness audits during final evaluation.
